In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/yashaswi0303/nasdaq-news/MSFT.csv
/kaggle/input/datasets/yashaswi0303/nasdaq-news/MDT.csv
/kaggle/input/datasets/yashaswi0303/nasdaq-news/AAPL.csv
/kaggle/input/datasets/yashaswi0303/nasdaq-news/AA.csv
/kaggle/input/datasets/yashaswi0303/nasdaq-news/AMZN.csv
/kaggle/input/datasets/yashaswi0303/nasdaq-news/KR.csv


In [6]:
# ================================
# Imports
# ================================

import pandas as pd
import torch
import os
import re
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

# ================================
# GPU setup
# ================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ================================
# Load FinBERT (only once)
# ================================

MODEL_NAME = "ProsusAI/finbert"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

model.to(device)
model.eval()

# ================================
# Text cleaning functions
# ================================

def clean_text(text):
    text = re.sub(r'<.*?>', '', str(text))
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def first_500_words(text):
    return " ".join(text.split()[:500])

# ================================
# Batch FinBERT sentiment
# ================================

def batch_sentiment(texts, batch_size=32):

    scores = []

    for i in tqdm(range(0, len(texts), batch_size)):

        batch = texts[i:i+batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        )

        inputs = {k:v.to(device) for k,v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        probs = F.softmax(outputs.logits, dim=1)

        neg = probs[:,0]
        pos = probs[:,2]

        batch_scores = (10 * pos) + (-5 * neg)

        scores.extend(batch_scores.cpu().numpy())

    return scores

# ================================
# Process one stock file
# ================================

def process_stock(file_path):

    print("\nProcessing:", file_path)

    df = pd.read_csv(file_path)

    df = df[["Date","Stock_symbol","Publisher","Article"]]

    df["Date"] = pd.to_datetime(df["Date"], utc=True)

    df = df.dropna(subset=["Article"])

    df["Article"] = df["Article"].apply(clean_text)

    df["Article_500"] = df["Article"].apply(first_500_words)

    df["Week"] = df["Date"].dt.to_period("W")

    df["Sentiment"] = batch_sentiment(df["Article_500"].tolist())

    weekly = df.groupby("Week").agg(
        Avg_Sentiment=("Sentiment","mean"),
        Article_Count=("Sentiment","count")
    ).reset_index()

    # Extract stock name from filename
    stock = os.path.basename(file_path).replace(".csv","")

    output_path = f"/kaggle/working/{stock}_weekly_sentiment.csv"

    weekly.to_csv(output_path, index=False)

    print("Saved:", output_path)


# ================================
# Process all CSV files
# ================================

import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    
    for filename in filenames:
        
        if filename.endswith(".csv"):
            
            file_path = os.path.join(dirname, filename)
            
            print("Processing:", file_path)
            
            process_stock(file_path)

Using device: cuda


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Processing: /kaggle/input/datasets/yashaswi0303/nasdaq-news/MSFT.csv

Processing: /kaggle/input/datasets/yashaswi0303/nasdaq-news/MSFT.csv


/tmp/ipykernel_55/3016089227.py:100: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["Week"] = df["Date"].dt.to_period("W")
100%|██████████| 274/274 [02:28<00:00,  1.85it/s]


...

Saved: /kaggle/working/AA_weekly_sentiment.csv
Processing: /kaggle/input/datasets/yashaswi0303/nasdaq-news/AMZN.csv

Processing: /kaggle/input/datasets/yashaswi0303/nasdaq-news/AMZN.csv


/tmp/ipykernel_55/3016089227.py:100: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["Week"] = df["Date"].dt.to_period("W")
100%|██████████| 150/150 [01:21<00:00,  1.83it/s]
/tmp/ipykernel_55/3016089227.py:100: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["Week"] = df["Date"].dt.to_period("W")


Saved: /kaggle/working/AMZN_weekly_sentiment.csv
Processing: /kaggle/input/datasets/yashaswi0303/nasdaq-news/KR.csv

Processing: /kaggle/input/datasets/yashaswi0303/nasdaq-news/KR.csv


0it [00:00, ?it/s]

Saved: /kaggle/working/KR_weekly_sentiment.csv
